In [0]:
%pip install torch optuna

In [0]:
"""
Baseline: raw CLMBR embeddings -> adapter -> linear classifier, NO prototype
layer. Adapter and the final fusion-style head are trained jointly, in a
single stage
"""

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, brier_score_loss, balanced_accuracy_score
)
import optuna

CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]

# ---- centralized hyperparameters ----
CFG = {
    "flags_path": "catalog.schema.step7_filtered_cohort_with_apos",
    "embeddings_path": "catalog.schema.patient_clmbr_embeddings_new_v5",  # global (raw) table
    "embedding_dim": 768,
    "adapter_hidden": 256,
    "adapter_out": 64,
    "n_classes": 3,
    "random_seed": 42,

    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 512,
    "max_epochs": 200,
    "patience": 15,

    "n_l1_trials": 20,
    "l1_search_low": 1e-6,
    "l1_search_high": 1e-1,

    "n_bootstrap": 1000,
    "bootstrap_confidence": 0.95,
}


# ============================================================================
# Split + data loading 
# ============================================================================

def build_person_split(cfg):
    flags_df = spark.table(cfg["flags_path"]).select(
        "person_id", "date_of_birth", "recorded_episode_start", "inferred_episode_start",
        "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control"
    ).toPandas()
    flags_df["person_id"] = flags_df["person_id"].astype(str)

    flags_df["y_control"] = flags_df["is_control"].astype(int)
    flags_df["y_pe_ghtn"] = ((flags_df["has_pe"] == 1) | (flags_df["has_inferred_pe"] == 1)
                              | (flags_df["has_ghtn"] == 1)).astype(int)
    flags_df["y_gdm"] = flags_df["has_gdm"].astype(int)

    for c in ["date_of_birth", "recorded_episode_start", "inferred_episode_start"]:
        flags_df[c] = pd.to_datetime(flags_df[c])
    flags_df["episode_start"] = flags_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)
    flags_df["age"] = (flags_df["episode_start"] - flags_df["date_of_birth"]).dt.days / 365.25

    combo = (flags_df["y_control"] * 4 + flags_df["y_pe_ghtn"] * 2 + flags_df["y_gdm"]).astype(int)
    pid_tv, pid_test = train_test_split(
        flags_df["person_id"].values, test_size=0.10, stratify=combo, random_state=cfg["random_seed"]
    )
    combo_tv = combo[flags_df["person_id"].isin(pid_tv)].values
    pid_train, pid_val = train_test_split(
        pid_tv, test_size=0.10 / 0.90, stratify=combo_tv, random_state=cfg["random_seed"]
    )
    return {"pid_train": set(pid_train), "pid_val": set(pid_val), "pid_test": set(pid_test),
            "flags_df": flags_df}


def load_global_data(cfg, split, device):
    df = spark.table(cfg["embeddings_path"]).toPandas()
    df["patient_id"] = df["patient_id"].astype(str)
    label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]
    df = df.merge(split["flags_df"][["person_id"] + label_cols + ["age"]],
                  left_on="patient_id", right_on="person_id", how="inner")

    emb_cols = sorted([c for c in df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    assert len(emb_cols) == cfg["embedding_dim"]

    out = {}
    for name, pid_set in [("train", split["pid_train"]), ("val", split["pid_val"]), ("test", split["pid_test"])]:
        sub = df[df["patient_id"].isin(pid_set)]
        out[f"X_{name}"] = sub[emb_cols].values.astype(np.float32)
        out[f"Y_{name}"] = sub[label_cols].values.astype(np.float32)
        out[f"pid_{name}"] = sub["patient_id"].values
        out[f"age_{name}"] = sub["age"].values.astype(np.float32)

    pos_freq = out["Y_train"].mean(axis=0).clip(0.03, 0.97)
    out["class_weights"] = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)
    print(f"[raw-embed baseline] train={len(out['X_train'])} val={len(out['X_val'])} test={len(out['X_test'])}")
    return out


# ============================================================================
# adapter -> concat age -> linear
# ============================================================================

class AdapterMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )

    def forward(self, x):
        return self.net(x)


class AdapterLinearHead(nn.Module):
    """adapter(x) concatenated with the age covariate, then a single linear
    layer"""
    def __init__(self, embedding_dim, adapter_hidden, adapter_out, n_classes, n_covariates=1):
        super().__init__()
        self.adapter = AdapterMLP(embedding_dim, adapter_hidden, adapter_out)
        self.classifier = nn.Linear(adapter_out + n_covariates, n_classes)

    def forward(self, x, covariates):
        z = self.adapter(x)
        combined = torch.cat([z, covariates], dim=1)
        return self.classifier(combined)

def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    results["auroc_macro"] = macro("auroc")
    results["auprc_macro"] = macro("auprc")
    results["f1_macro"] = macro("f1")
    results["bacc_macro"] = macro("bacc")
    results["brier_macro"] = macro("brier")

    out_auroc = [results[f"auroc_{n}"] for n in outcome_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro_outcomes_only"] = float(np.mean(out_auroc)) if out_auroc else float("nan")
    out_auprc = [results[f"auprc_{n}"] for n in outcome_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro_outcomes_only"] = float(np.mean(out_auprc)) if out_auprc else float("nan")

    results["_primary"] = results["auroc_macro"]
    return results


# ============================================================================
# Training: joint adapter+linear, BCE + L1 penalty on classifier weight only
# (bias excluded), 20-trial Optuna sweep over lambda_l1 on val macro AUROC
# ============================================================================

def train_adapter_linear(X_train, age_train, Y_train, X_val, age_val, Y_val,
                          cfg, class_weights, device, lambda_l1=0.0, verbose=True):
    model = AdapterLinearHead(cfg["embedding_dim"], cfg["adapter_hidden"], cfg["adapter_out"],
                              cfg["n_classes"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)

    age_train_n = (age_train - age_train.mean()) / (age_train.std() + 1e-8)
    age_val_n = (age_val - age_train.mean()) / (age_train.std() + 1e-8)  # normalize val w/ train stats, no leakage

    ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(age_train_n.reshape(-1, 1), dtype=torch.float32),
        torch.tensor(Y_train, dtype=torch.float32),
    )
    loader = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True)

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    age_val_t = torch.tensor(age_val_n.reshape(-1, 1), dtype=torch.float32).to(device)

    best_auroc, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        for xb, ab, yb in loader:
            xb, ab, yb = xb.to(device), ab.to(device), yb.to(device)
            logits = model(xb, ab)
            loss = loss_fn(logits, yb)
            if lambda_l1 > 0:
                loss = loss + lambda_l1 * model.classifier.weight.abs().sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            p_val = torch.sigmoid(model(X_val_t, age_val_t)).cpu().numpy()
        macro_auroc = _compute_metric_suite(Y_val, p_val)["auroc_macro"]

        if macro_auroc > best_auroc:
            best_auroc = macro_auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f"  adapter-linear ep {epoch:3d} | val macro AUROC {macro_auroc:.4f} | pat {patience_ctr}/{cfg['patience']}")
        if patience_ctr >= cfg["patience"]:
            break

    model.load_state_dict(best_state)
    return model, best_auroc


def sweep_lambda_l1(X_train, age_train, Y_train, X_val, age_val, Y_val, cfg, class_weights, device):
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=cfg["random_seed"]),
                                study_name="raw_embed_adapter_linear_l1")

    def objective(trial):
        lambda_l1 = trial.suggest_float("lambda_l1", cfg["l1_search_low"], cfg["l1_search_high"], log=True) \
            if trial.number > 0 else 0.0
        _, val_auroc = train_adapter_linear(
            X_train, age_train, Y_train, X_val, age_val, Y_val,
            cfg, class_weights, device, lambda_l1=lambda_l1, verbose=False,
        )
        trial.set_user_attr("lambda_l1", lambda_l1)
        return val_auroc

    study.optimize(objective, n_trials=cfg["n_l1_trials"], show_progress_bar=False)
    best_lambda_l1 = study.best_trial.user_attrs["lambda_l1"]
    print(f"Best lambda_l1: {best_lambda_l1:.6g}  (val macro AUROC {study.best_value:.4f})")
    return best_lambda_l1

def bootstrap_test_metrics_ci(Y_true, Y_prob, class_names, n_bootstrap=1000, confidence=0.95, seed=42):
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    macro_keys = ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
                 "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
    per_class_keys = [f"{m}_{name}" for m in ["auroc", "auprc", "f1", "bacc", "brier"] for name in class_names]
    all_keys = macro_keys + per_class_keys
    boot_scores = {k: [] for k in all_keys}

    n_skipped = 0
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt_r, yp_r = Y_true[idx], Y_prob[idx]
        degenerate = any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names)))
        if degenerate:
            n_skipped += 1
            continue
        m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
        for k in all_keys:
            boot_scores[k].append(m[k])

    results = {}
    for k in all_keys:
        arr = np.array(boot_scores[k])
        arr = arr[~np.isnan(arr)]
        if len(arr) == 0:
            results[k] = (float("nan"), float("nan"), float("nan"))
            continue
        lo, hi = np.percentile(arr, [(1 - confidence) / 2 * 100, (1 + confidence) / 2 * 100])
        results[k] = (arr.mean(), lo, hi)

    if n_skipped > 0:
        print(f"  ({n_skipped}/{n_bootstrap} resamples skipped -- degenerate label in that draw)")
    return results

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    split = build_person_split(CFG)  # same seed/flags_path -> identical pid_train/val/test as fusion notebook
    data = load_global_data(CFG, split, device)

    print("\n=== Sweeping lambda_l1 (adapter + linear, raw embeddings, no prototype layer) ===")
    best_lambda_l1 = sweep_lambda_l1(
        data["X_train"], data["age_train"], data["Y_train"],
        data["X_val"], data["age_val"], data["Y_val"],
        CFG, data["class_weights"], device,
    )

    print("\n=== Final fit at best lambda_l1 ===")
    model, val_auroc = train_adapter_linear(
        data["X_train"], data["age_train"], data["Y_train"],
        data["X_val"], data["age_val"], data["Y_val"],
        CFG, data["class_weights"], device, lambda_l1=best_lambda_l1,
    )
    print(f"Final val macro AUROC: {val_auroc:.4f}")

    # --- held-out test evaluation (touched once) ---
    model.eval()
    age_test_n = (data["age_test"] - data["age_train"].mean()) / (data["age_train"].std() + 1e-8)
    with torch.no_grad():
        X_test_t = torch.tensor(data["X_test"], dtype=torch.float32).to(device)
        age_test_t = torch.tensor(age_test_n.reshape(-1, 1), dtype=torch.float32).to(device)
        p_test = torch.sigmoid(model(X_test_t, age_test_t)).cpu().numpy()

    test_metrics = _compute_metric_suite(data["Y_test"], p_test)

    print("\n=== FINAL HELD-OUT TEST METRICS (raw embeddings -> adapter -> linear, fusion-style head) ===")
    for key in ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
               "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]:
        print(f"  {key:<32} {test_metrics[key]:.4f}")

    print("\nPer-class test metrics:")
    for name in CLASS_NAMES:
        print(f"  {name:10s}  AUROC {test_metrics[f'auroc_{name}']:.4f}  "
              f"AUPRC {test_metrics[f'auprc_{name}']:.4f}  F1 {test_metrics[f'f1_{name}']:.4f}  "
              f"BalAcc {test_metrics[f'bacc_{name}']:.4f}  Brier {test_metrics[f'brier_{name}']:.4f}")

    print("\n" + "=" * 70)
    print(f"BOOTSTRAPPED {int(CFG['bootstrap_confidence']*100)}% CIs ({CFG['n_bootstrap']} resamples, held-out test)")
    print("=" * 70)
    ci = bootstrap_test_metrics_ci(data["Y_test"], p_test, CLASS_NAMES,
                                   n_bootstrap=CFG["n_bootstrap"], confidence=CFG["bootstrap_confidence"],
                                   seed=CFG["random_seed"])

    print("\nMacro metrics:")
    for key in ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
               "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]:
        mean, lo, hi = ci[key]
        print(f"  {key:<32} {mean:.4f}  (95% CI [{lo:.4f}, {hi:.4f}])")

    print("\nPer-class metrics:")
    for name in CLASS_NAMES:
        print(f"  {name}:")
        for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
            mean, lo, hi = ci[f"{m}_{name}"]
            print(f"    {m:8s} {mean:.4f}  (95% CI [{lo:.4f}, {hi:.4f}])")

    # --- export test predictions for cross-baseline paired bootstrap ---
    import pickle, os
    EXPORT_DIR = "/path/to/exports"
    os.makedirs(EXPORT_DIR, exist_ok=True)
    EXPORT_PATH = os.path.join(EXPORT_DIR, "protopregnet_non_interp_test_predictions.pkl")
    payload = {
        "source": "protopregnet_non_interp_baseline",
        "class_names": CLASS_NAMES,
        "person_id": np.asarray(data["pid_test"]).astype(str),
        "Y_true": np.asarray(data["Y_test"]),
        "predictions": {
            "Adapter+Linear (non-interp)": np.asarray(p_test),
        },
    }
    with open(EXPORT_PATH, "wb") as f:
        pickle.dump(payload, f)
    print(f"\nExported {len(payload['person_id'])} test patients -> {EXPORT_PATH}")
    print("(Paired significance vs ProtoPregNet: see protopregnet_vs_baselines_significance.ipynb)")